# DCT/IDCT on Arty
Uses the exact fixed-point reference generated from the supplied Lena block.

In [ ]:
import time
import numpy as np
from pynq import Overlay
overlay = Overlay("../exported/lab3.bit")
ip = overlay.dct_accel_0
reference = np.load("dct_reference.npz")
print(ip.register_map)

In [ ]:
def transform(data, inverse=0):
    for i,x in enumerate(data):
        ip.write(0x100+4*i, int(x) & 0xffffffff)
    ip.write(0x10, inverse)
    ip.write(0, 1)
    deadline = time.monotonic()+1
    while not (ip.read(0) & 2):
        if time.monotonic()>deadline:
            raise TimeoutError("DCT ap_done timeout")
    result = np.array([ip.read(0x200+4*i) for i in range(64)],dtype=np.uint32).view(np.int32)
    return result
forward = transform(reference["input"])
recovered = transform(forward, 1)
np.testing.assert_array_equal(forward, reference["forward"])
np.testing.assert_array_equal(recovered, reference["inverse"])
print("DCT/IDCT PASS")
print(recovered.reshape(8,8))